# Workflow Plan: Structured + Unstructured Fraud/Anomaly Detection

**Objective:** Demonstrate that combining structured financial features (`train_transaction` + `train_identity`) with unstructured text signals (`synthetic_crypto_fraud_narratives`) improves anomaly/fraud detection over structured features alone.

---

## Data Landscape

| Dataset | Rows | Key Characteristics |
| --- | --- | --- |
| `hackathon.shared_datasets.train_transaction` | 590,540 | 3.5% fraud; 394 features (amounts, card, email, C/D/V/M columns) |
| `hackathon.shared_datasets.train_identity` | 144,233 | Covers ~24% of transactions; device/network identity signals |
| `hackathon.shared_datasets.synthetic_crypto_fraud_narratives` | 1,000 | 60% fraud / 40% legit; free-text narratives + numeric crypto indicators |

---

## UK Government AI Playbook — Principles Mapped to Workflow

This plan integrates the 10 key principles from the [UK Government AI Playbook](https://www.gov.uk/government/publications/ai-playbook-for-the-uk-government/artificial-intelligence-playbook-for-the-uk-government-html).

---

### Phase 0 — Problem Definition & Proportionality
*Playbook Principles: "Start with clear user need", "Be proportionate", "Justify the use of AI"*

- [ ] Define the detection goal: flag suspicious transactions for human review (not auto-block)
- [ ] Document why AI is appropriate: high-dimensional feature space (394+ cols), non-linear fraud patterns, volume too high for manual review
- [ ] Identify proportionality: semi-supervised anomaly detection augments (does not replace) human decision-making
- [ ] Define success metrics: PR-AUC, Recall@FPR=5%, F1 on fraud class; improvement over structured-only baseline

---

### Phase 1 — Data Assessment & Ethics Review
*Playbook Principles: "Use good data", "Be fair", "Consider ethics early"*

- [ ] **Join structured data**: LEFT JOIN `train_transaction` ↔ `train_identity` on `TransactionID` (expect 24% match rate)
- [ ] **Profile missingness**: identity features will be NULL for 76% of transactions — plan imputation/indicator strategy
- [ ] **Assess bias risks**: check if fraud rate differs by `DeviceType`, `card4`/`card6`, `P_emaildomain` — flag any protected-characteristic proxies
- [ ] **Data quality checks**: duplicate TransactionIDs, timestamp integrity, distribution drift between joined vs. unjoined cohorts
- [ ] **Narratives provenance**: document that crypto narratives are synthetic (not real PII); assess whether patterns transfer to the financial fraud domain

---

### Phase 2 — Feature Engineering
*Playbook Principles: "Iterate and learn", "Use data responsibly"*

#### 2a. Structured Features (from `train_transaction` + `train_identity`)
- [ ] Select high-signal columns: `TransactionAmt`, `C1-C14`, `D1-D15`, `V1-V45`, `ProductCD`, `card4/6`, `addr1/2`
- [ ] Identity enrichment: `DeviceType`, `DeviceInfo`, `id_01`–`id_38` (where available)
- [ ] Derived features: amt-to-card-mean ratio, velocity features (txns per card in last N), email domain risk scoring
- [ ] Handle categoricals: one-hot or target-encoding for `ProductCD`, `card4`, `card6`, email domains
- [ ] Handle missingness: "is_missing" indicator columns + median imputation for numerics

#### 2b. Unstructured Features (from `synthetic_crypto_fraud_narratives`)
- [ ] **Text embeddings**: generate vector representations of `narrative` field using Databricks Foundation Model API (`ai_query` with an embedding model) or sentence-transformers
- [ ] **NLP-derived signals**: extract keyword indicators ("coordinated", "liquidity withdrawn", "newly active wallets"), sentiment polarity, narrative length, entity counts
- [ ] **Structured crypto indicators**: `price_change_pct`, `volume_spike_x`, `social_mention_spike_x`, `top10_holder_pct`, `unique_buy_wallets`
- [ ] Dimensionality reduction on embeddings: PCA or UMAP to ~50 components

---

### Phase 3 — Modelling (Iterative, with Baselines)
*Playbook Principles: "Start small and iterate", "Test rigorously"*

#### Experiment A — Structured-Only Baseline
- [ ] Train XGBoost/LightGBM on joined `train_transaction` + `train_identity` features
- [ ] Address class imbalance: `scale_pos_weight`, SMOTE, or focal loss
- [ ] Log to MLflow: hyperparameters, metrics (ROC-AUC, PR-AUC, Fraud-F1, Recall@5%FPR)

#### Experiment B — Unstructured-Only (Crypto Narratives)
- [ ] Train classifier on `synthetic_crypto_fraud_narratives` using text embeddings + numeric crypto features
- [ ] Establishes whether narrative text alone carries discriminatory power
- [ ] Log to same MLflow experiment for comparison

#### Experiment C — Combined / Augmented Model
- [ ] **Strategy 1 — Feature fusion**: concatenate structured features with text-derived features for the crypto dataset
- [ ] **Strategy 2 — Transfer learning**: train a text-based fraud-pattern detector on narratives, then use its confidence scores as an auxiliary feature in the main structured model
- [ ] **Strategy 3 — Anomaly ensemble**: train isolation forest on structured data + separately on embeddings; combine anomaly scores
- [ ] Compare all experiments on held-out test set; use statistical significance testing (McNemar's or DeLong's test on AUCs)

---

### Phase 4 — Explainability & Transparency
*Playbook Principles: "Be transparent", "Explain decisions", "Maintain human oversight"*

- [ ] SHAP values for structured model — identify top risk drivers per prediction
- [ ] Attention/token-importance for text features — which narrative phrases trigger fraud signals
- [ ] Generate human-readable "risk cards" for flagged transactions (top 3 reasons + confidence)
- [ ] Define escalation thresholds: high-confidence → auto-flag; medium → analyst queue; low → pass

---

### Phase 5 — Fairness & Bias Testing
*Playbook Principles: "Be fair", "Consider and address potential harms"*

- [ ] Evaluate model performance across demographic proxies (`DeviceType`, `card4`, email domain categories)
- [ ] Check for disparate false-positive rates across groups
- [ ] Document any fairness trade-offs and mitigation steps taken
- [ ] Test on the crypto narratives: does model equally detect different fraud typologies (pump-and-dump, liquidity rug, coordinated sell)?

---

### Phase 6 — Governance, Accountability & Deployment Readiness
*Playbook Principles: "Be accountable", "Manage risks", "Use AI legally"*

- [ ] Register final model in Unity Catalog Model Registry with version, lineage, and metadata tags
- [ ] Document model card: intended use, limitations, known biases, performance boundaries
- [ ] Define monitoring plan: data drift detection, performance decay alerts, retraining triggers
- [ ] RACI matrix: who approves model updates, who reviews flagged cases, who owns data quality
- [ ] Audit trail: MLflow experiment tracking provides full reproducibility of all decisions

---

### Phase 7 — Share, Collaborate & Iterate
*Playbook Principles: "Share and collaborate", "Build internal capability"*

- [ ] Publish findings to team dashboard (structured-only vs. combined performance comparison)
- [ ] Document lessons learned: what worked, what didn't, where unstructured data added most value
- [ ] Propose next iteration: real-time scoring pipeline, additional unstructured sources (transaction memos, customer comms)

---

## Expected Outcome

| Metric | Structured Only (X) | Structured + Unstructured (Y) |
| --- | --- | --- |
| ROC-AUC | Baseline | Target: +2-5% |
| PR-AUC | Baseline | Target: +5-10% |
| Fraud Recall @5% FPR | Baseline | Target: +10-15% |
| Explainability | Feature importance only | Feature importance + narrative reasoning |

**Hypothesis:** Unstructured signals capture fraud *intent* and *coordination* patterns that structured features alone cannot encode, particularly for novel/emerging fraud typologies.

In [0]:

spark.table("hackathon.shared_datasets.train_transaction").display()

## Phase 1 — Data Assessment & Ethics Review
*UK AI Playbook Principles: "Use good data", "Be fair", "Consider ethics early"*

This phase joins the structured datasets, profiles data quality and missingness, assesses potential bias risks, and documents the provenance of the unstructured narratives data.

In [0]:
# ============================================================
# 1.1 Load and Join Structured Data
# LEFT JOIN train_transaction with train_identity on TransactionID
# ============================================================

df_transaction = spark.table("hackathon.shared_datasets.train_transaction")
df_identity = spark.table("hackathon.shared_datasets.train_identity")

# LEFT JOIN — keeps all transactions, enriches with identity where available
df_joined = df_transaction.join(df_identity, on="TransactionID", how="left")

# Summary statistics
txn_count = df_transaction.count()
id_count = df_identity.count()
joined_count = df_joined.count()
identity_match_rate = id_count / txn_count * 100

print(f"Transactions:        {txn_count:,}")
print(f"Identity records:    {id_count:,}")
print(f"Joined dataset:      {joined_count:,} rows")
print(f"Identity match rate: {identity_match_rate:.1f}%")
print(f"Columns after join:  {len(df_joined.columns)}")
print(f"\nFraud distribution:")
df_joined.groupBy("isFraud").count().orderBy("isFraud").display()

In [0]:
# ============================================================
# 1.2 Profile Missingness
# Calculate null percentage for every column in the joined dataset
# Identity features expected ~76% null (only 24% of txns have identity)
# ============================================================
from pyspark.sql import functions as F

total_rows = df_joined.count()

# Calculate null counts for all columns
null_counts = df_joined.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_joined.columns]
)

# Convert to a readable format
import pandas as pd

null_row = null_counts.collect()[0]
null_df = pd.DataFrame({
    "column": df_joined.columns,
    "null_count": [null_row[c] for c in df_joined.columns],
    "null_pct": [null_row[c] / total_rows * 100 for c in df_joined.columns]
}).sort_values("null_pct", ascending=False)

# Mark source table for each column
identity_cols = set(df_identity.columns) - {"TransactionID"}
null_df["source"] = null_df["column"].apply(
    lambda c: "identity" if c in identity_cols else "transaction"
)

print(f"Total columns: {len(null_df)}")
print(f"Columns with >50% null: {(null_df['null_pct'] > 50).sum()}")
print(f"Columns with 0% null: {(null_df['null_pct'] == 0).sum()}")
print(f"\n--- Top 30 columns by missingness ---")
display(spark.createDataFrame(null_df.head(30)))

In [0]:
# ============================================================
# 1.3 Bias Risk Assessment
# Check fraud rate by key categorical features that could
# serve as proxies for protected characteristics
# UK AI Playbook: "Be fair", "Consider and address potential harms"
# ============================================================

print("="*60)
print("BIAS RISK ASSESSMENT")
print("Checking fraud rate disparity across demographic-proxy features")
print("="*60)

# --- Fraud rate by card type (card4) ---
print("\n📊 Fraud rate by card4 (card brand):")
df_joined.groupBy("card4").agg(
    F.count("*").alias("total"),
    F.sum("isFraud").alias("fraud_count"),
    F.round(F.mean("isFraud") * 100, 2).alias("fraud_rate_pct")
).orderBy(F.desc("total")).display()

# --- Fraud rate by card category (card6) ---
print("\n📊 Fraud rate by card6 (debit/credit):")
df_joined.groupBy("card6").agg(
    F.count("*").alias("total"),
    F.sum("isFraud").alias("fraud_count"),
    F.round(F.mean("isFraud") * 100, 2).alias("fraud_rate_pct")
).orderBy(F.desc("total")).display()

# --- Fraud rate by email domain (top 10) ---
print("\n📊 Fraud rate by P_emaildomain (top 10 by volume):")
df_joined.groupBy("P_emaildomain").agg(
    F.count("*").alias("total"),
    F.sum("isFraud").alias("fraud_count"),
    F.round(F.mean("isFraud") * 100, 2).alias("fraud_rate_pct")
).orderBy(F.desc("total")).limit(10).display()

# --- Fraud rate by DeviceType (from identity) ---
print("\n📊 Fraud rate by DeviceType:")
df_joined.groupBy("DeviceType").agg(
    F.count("*").alias("total"),
    F.sum("isFraud").alias("fraud_count"),
    F.round(F.mean("isFraud") * 100, 2).alias("fraud_rate_pct")
).orderBy(F.desc("total")).display()

print("\n⚠️  FAIRNESS NOTE: Large disparities above may indicate")
print("   features that act as proxies for protected characteristics.")
print("   These should be reviewed before inclusion in the final model.")

In [0]:
# ============================================================
# 1.4 Data Quality Checks
# - Duplicate TransactionIDs
# - Timestamp integrity
# - Distribution drift: joined vs unjoined cohorts
# ============================================================

print("="*60)
print("DATA QUALITY CHECKS")
print("="*60)

# --- Check 1: Duplicate TransactionIDs ---
total_txns = df_transaction.count()
unique_txns = df_transaction.select("TransactionID").distinct().count()
duplicates = total_txns - unique_txns
print(f"\n✅ Duplicate TransactionIDs: {duplicates:,} (of {total_txns:,} total)")

# --- Check 2: Timestamp integrity ---
print("\n📊 TransactionDT (timestamp offset) distribution:")
df_transaction.select(
    F.min("TransactionDT").alias("min_dt"),
    F.max("TransactionDT").alias("max_dt"),
    F.mean("TransactionDT").alias("mean_dt"),
    F.stddev("TransactionDT").alias("stddev_dt")
).display()

# --- Check 3: Distribution drift — joined vs unjoined cohorts ---
print("\n📊 Fraud rate comparison: transactions WITH vs WITHOUT identity match")
df_joined.withColumn(
    "has_identity", F.when(F.col("DeviceType").isNotNull(), "Yes").otherwise("No")
).groupBy("has_identity").agg(
    F.count("*").alias("total"),
    F.sum("isFraud").alias("fraud_count"),
    F.round(F.mean("isFraud") * 100, 2).alias("fraud_rate_pct"),
    F.round(F.mean("TransactionAmt"), 2).alias("avg_txn_amt")
).display()

# --- Check 4: TransactionAmt anomalies ---
print("\n📊 TransactionAmt distribution (check for outliers):")
df_transaction.select(
    F.min("TransactionAmt").alias("min"),
    F.expr("percentile_approx(TransactionAmt, 0.25)").alias("p25"),
    F.expr("percentile_approx(TransactionAmt, 0.50)").alias("median"),
    F.expr("percentile_approx(TransactionAmt, 0.75)").alias("p75"),
    F.expr("percentile_approx(TransactionAmt, 0.99)").alias("p99"),
    F.max("TransactionAmt").alias("max")
).display()

In [0]:
# ============================================================
# 1.5 Narratives Provenance & Transferability Assessment
# UK AI Playbook: "Use good data" — document synthetic origin,
# assess whether patterns are transferable to financial fraud
# ============================================================

df_narratives = spark.table("hackathon.shared_datasets.synthetic_crypto_fraud_narratives")

print("="*60)
print("UNSTRUCTURED DATA: SYNTHETIC CRYPTO FRAUD NARRATIVES")
print("="*60)

# Basic profile
print(f"\nRows: {df_narratives.count():,}")
print(f"Columns: {df_narratives.columns}")

# Label distribution
print("\n📊 Label distribution:")
df_narratives.groupBy("label").agg(
    F.count("*").alias("count"),
    F.round(F.mean("price_change_pct"), 1).alias("avg_price_change_pct"),
    F.round(F.mean("volume_spike_x"), 1).alias("avg_volume_spike_x"),
    F.round(F.mean("social_mention_spike_x"), 1).alias("avg_social_spike_x"),
    F.round(F.mean("top10_holder_pct"), 1).alias("avg_top10_holder_pct")
).display()

# Narrative text characteristics
print("\n📊 Narrative text statistics:")
df_narratives.withColumn("narrative_length", F.length("narrative")).select(
    F.min("narrative_length").alias("min_chars"),
    F.round(F.mean("narrative_length"), 0).alias("avg_chars"),
    F.max("narrative_length").alias("max_chars")
).display()

# Sample narratives
print("\n📝 Sample FRAUD narratives:")
df_narratives.filter(F.col("label") == "fraud").select("narrative").limit(3).display()

print("\n📝 Sample LEGITIMATE narratives:")
df_narratives.filter(F.col("label") == "legitimate").select("narrative").limit(3).display()

print("\n" + "="*60)
print("PROVENANCE DOCUMENTATION")
print("="*60)
print("""
• Source: Synthetic dataset (not real customer data)
• PII Risk: NONE — no real personally identifiable information
• Purpose: Demonstrate that NLP features from free-text fraud
  descriptions can augment structured feature models
• Transferability rationale: Fraud narratives encode behavioural
  patterns (coordinated activity, liquidity manipulation, social
  engineering) that are domain-transferable to financial fraud
• Limitation: Small sample (1,000 rows) — results should be
  interpreted as proof-of-concept, not production-grade
""")

In [0]:
# ============================================================
# 1.6 Phase 1 Summary & Recommendations
# ============================================================

print("="*60)
print("PHASE 1 SUMMARY — DATA ASSESSMENT & ETHICS REVIEW")
print("="*60)
print(f"""
📋 KEY FINDINGS:

1. JOINED DATASET
   • {joined_count:,} transactions with {len(df_joined.columns)} features after join
   • Identity enrichment available for {identity_match_rate:.1f}% of transactions
   • Fraud rate: {20663/590540*100:.2f}% (heavily imbalanced)

2. MISSINGNESS STRATEGY NEEDED
   • Identity columns ~76% null → use "has_identity" indicator + impute
   • Some V-columns have >80% nulls → consider dropping or flag-based approach

3. BIAS RISKS IDENTIFIED
   • Review card type and email domain fraud rate disparities
   • DeviceType shows differential fraud rates — document and monitor
   • Recommendation: test model fairness across these groups in Phase 5

4. DATA QUALITY
   • No duplicate TransactionIDs — clean primary key
   • Transaction amounts show heavy right skew — log-transform candidate
   • Cohort drift: transactions with identity match may have different
     fraud characteristics than those without

5. UNSTRUCTURED DATA (NARRATIVES)
   • 1,000 synthetic records (600 fraud / 400 legit)
   • Rich text describing fraud coordination patterns
   • No PII concerns — safe to use for NLP feature extraction
   • Small sample → proof-of-concept scope

✅ READY FOR PHASE 2: Feature Engineering
""")

In [0]:
# ============================================================
# Save joined structured data and narratives as Delta tables
# Delta Lake provides ACID transactions, time travel, and
# efficient reads for downstream modelling phases
# ============================================================

# 1. Save the joined transaction + identity dataset
(
    df_joined
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hackathon.shared_datasets.fraud_busters_train_joined")
)
print("✅ Saved: hackathon.shared_datasets.fraud_busters_train_joined")
print(f"   Rows: {spark.table('hackathon.shared_datasets.fraud_busters_train_joined').count():,}")
print(f"   Columns: {len(spark.table('hackathon.shared_datasets.fraud_busters_train_joined').columns)}")

# 2. Save the narratives (already small, but Delta gives us versioning)
(
    df_narratives
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hackathon.shared_datasets.fraud_busters_narratives_enriched")
)
print("\n✅ Saved: hackathon.shared_datasets.fraud_busters_narratives_enriched")
print(f"   Rows: {spark.table('hackathon.shared_datasets.fraud_busters_narratives_enriched').count():,}")
print(f"   Columns: {len(spark.table('hackathon.shared_datasets.fraud_busters_narratives_enriched').columns)}")

print("\n📋 Both tables now available as managed Delta tables with:")
print("   • ACID transactions")
print("   • Time travel (VERSION AS OF / TIMESTAMP AS OF)")
print("   • Schema enforcement")
print("   • Optimized reads via data skipping")